In [1]:
# ── STEP 1: Install and Import Libraries ────────────────────
!pip install yfinance pandas-datareader --quiet

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# STEP 2: Download BIST 100 and S&P 500 Data
# ============================================================

START_DATE = "2010-01-01"
END_DATE   = datetime.today().strftime('%Y-%m-%d')

print(f"\n📥 Downloading data: {START_DATE} → {END_DATE}")

# BIST 100
bist100 = yf.download("XU100.IS", start=START_DATE, end=END_DATE)
bist100 = bist100[['Close']].rename(columns={'Close': 'BIST100_Close'})
print(f"✅ BIST 100: {len(bist100)} rows downloaded.")

# S&P 500
sp500 = yf.download("^GSPC", start=START_DATE, end=END_DATE)
sp500 = sp500[['Close']].rename(columns={'Close': 'SP500_Close'})
print(f"✅ S&P 500: {len(sp500)} rows downloaded.")


📥 Downloading data: 2010-01-01 → 2026-04-14


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

✅ BIST 100: 4080 rows downloaded.
✅ S&P 500: 4093 rows downloaded.


In [4]:
# ============================================================
# STEP 3: Download FRED-MD Macroeconomic Data
# ============================================================

# Alternative FRED-MD URL (direct access)
fred_url = "https://s3.amazonaws.com/files.fred.stlouisfed.org/fred-md/monthly/current.csv"

try:
    fred_raw = pd.read_csv(fred_url)
    # Skip the first row which contains transformation codes
    fred_md = fred_raw.iloc[1:].copy()
    fred_md = fred_md.rename(columns={fred_md.columns[0]: 'Date'})
    fred_md['Date'] = pd.to_datetime(fred_md['Date'])
    fred_md = fred_md.set_index('Date')
    fred_md = fred_md.apply(pd.to_numeric, errors='coerce')
    print(f"\n✅ FRED-MD: {len(fred_md)} rows, {len(fred_md.columns)} variables downloaded.")
    print(f"   Date range: {fred_md.index[0].date()} → {fred_md.index[-1].date()}")
    print(f"   Columns (first 10): {list(fred_md.columns[:10])}")

except Exception as e:
    print(f"⚠️ FRED-MD could not be downloaded: {e}")
    # Second fallback: download manually and upload to Colab
    print("\n📌 Manual fallback instructions:")
    print("   1. Go to: https://research.stlouisfed.org/econ/mccracken/fred-databases/")
    print("   2. Download the 'Monthly Database' CSV")
    print("   3. Upload it to Colab using the code below:")
    print("""
    from google.colab import files
    uploaded = files.upload()
    fred_raw = pd.read_csv(list(uploaded.keys())[0])
    fred_md = fred_raw.iloc[1:].copy()
    fred_md = fred_md.rename(columns={fred_md.columns[0]: 'Date'})
    fred_md['Date'] = pd.to_datetime(fred_md['Date'])
    fred_md = fred_md.set_index('Date')
    fred_md = fred_md.apply(pd.to_numeric, errors='coerce')
    print(f"✅ FRED-MD loaded manually: {len(fred_md)} rows")
    """)

⚠️ FRED-MD could not be downloaded: HTTP Error 403: Forbidden

📌 Manual fallback instructions:
   1. Go to: https://research.stlouisfed.org/econ/mccracken/fred-databases/
   2. Download the 'Monthly Database' CSV
   3. Upload it to Colab using the code below:

    from google.colab import files
    uploaded = files.upload()
    fred_raw = pd.read_csv(list(uploaded.keys())[0])
    fred_md = fred_raw.iloc[1:].copy()
    fred_md = fred_md.rename(columns={fred_md.columns[0]: 'Date'})
    fred_md['Date'] = pd.to_datetime(fred_md['Date'])
    fred_md = fred_md.set_index('Date')
    fred_md = fred_md.apply(pd.to_numeric, errors='coerce')
    print(f"✅ FRED-MD loaded manually: {len(fred_md)} rows")
    


In [5]:
from google.colab import files

uploaded = files.upload()

Saving 2026-03-MD.csv to 2026-03-MD.csv


In [6]:
fred_raw = pd.read_csv(list(uploaded.keys())[0])

# Skip the first row which contains transformation codes
fred_md = fred_raw.iloc[1:].copy()
fred_md = fred_md.rename(columns={fred_md.columns[0]: 'Date'})
fred_md['Date'] = pd.to_datetime(fred_md['Date'])
fred_md = fred_md.set_index('Date')
fred_md = fred_md.apply(pd.to_numeric, errors='coerce')

print(f"✅ FRED-MD loaded: {len(fred_md)} rows")
print(f"   Date range: {fred_md.index[0].date()} → {fred_md.index[-1].date()}")
print(f"   Number of variables: {len(fred_md.columns)}")

✅ FRED-MD loaded: 806 rows
   Date range: 1959-01-01 → 2026-02-01
   Number of variables: 126


In [8]:
# ============================================================
# STEP 4: Date Alignment and Missing Value Handling (Fixed)
# ============================================================

# Fix MultiIndex issue from yfinance
bist100_fixed = bist100.copy()
sp500_fixed = sp500.copy()

# Flatten column levels if MultiIndex exists
if isinstance(bist100_fixed.columns, pd.MultiIndex):
    bist100_fixed.columns = ['BIST100_Close']
if isinstance(sp500_fixed.columns, pd.MultiIndex):
    sp500_fixed.columns = ['SP500_Close']

# Merge BIST 100 and S&P 500 into a single daily DataFrame
daily_df = pd.concat([bist100_fixed, sp500_fixed], axis=1)
daily_df.index = pd.to_datetime(daily_df.index)

# Forward-fill to handle missing values (e.g. holidays)
daily_df = daily_df.ffill()
daily_df = daily_df.dropna()

print(f"✅ Combined daily data: {len(daily_df)} rows")
print(f"   Date range: {daily_df.index[0].date()} → {daily_df.index[-1].date()}")
print(f"   Columns: {list(daily_df.columns)}")

# Select key macroeconomic variables from FRED-MD
# CPIAUCSL: Consumer Price Index (inflation proxy)
# FEDFUNDS: Federal Funds Rate (monetary policy)
# INDPRO  : Industrial Production Index (economic activity)
fred_selected = fred_md[['CPIAUCSL', 'FEDFUNDS', 'INDPRO']].copy()

# Resample monthly FRED-MD data to daily frequency using forward-fill
fred_daily = fred_selected.resample('D').ffill()
fred_daily.index = pd.to_datetime(fred_daily.index)

# Merge daily price data with macroeconomic indicators
final_df = daily_df.join(fred_daily, how='left')
final_df = final_df.ffill().dropna()

print(f"\n✅ Final merged dataset (Price + Macro):")
print(f"   Number of rows : {len(final_df)}")
print(f"   Columns        : {list(final_df.columns)}")
print(f"   Date range     : {final_df.index[0].date()} → {final_df.index[-1].date()}")

✅ Combined daily data: 4223 rows
   Date range: 2010-01-04 → 2026-04-13
   Columns: ['BIST100_Close', 'SP500_Close']

✅ Final merged dataset (Price + Macro):
   Number of rows : 4223
   Columns        : ['BIST100_Close', 'SP500_Close', 'CPIAUCSL', 'FEDFUNDS', 'INDPRO']
   Date range     : 2010-01-04 → 2026-04-13


In [9]:
# ============================================================
# STEP 5: Save All Datasets to CSV
# ============================================================

final_df.to_csv("financial_data.csv")
bist100_fixed.to_csv("bist100_raw.csv")
sp500_fixed.to_csv("sp500_raw.csv")
fred_md.to_csv("fred_md_raw.csv")

print("\n✅ All datasets saved as CSV files:")
print("   📄 financial_data.csv  → Final merged dataset")
print("   📄 bist100_raw.csv     → Raw BIST 100 data")
print("   📄 sp500_raw.csv       → Raw S&P 500 data")
print("   📄 fred_md_raw.csv     → Raw FRED-MD data")


✅ All datasets saved as CSV files:
   📄 financial_data.csv  → Final merged dataset
   📄 bist100_raw.csv     → Raw BIST 100 data
   📄 sp500_raw.csv       → Raw S&P 500 data
   📄 fred_md_raw.csv     → Raw FRED-MD data


In [10]:
# ============================================================
# STEP 6: Summary Statistics
# ============================================================

print("\n" + "="*50)
print("DATASET SUMMARY")
print("="*50)
print(final_df.describe().round(4))


DATASET SUMMARY
       BIST100_Close  SP500_Close   CPIAUCSL   FEDFUNDS     INDPRO
count      4223.0000    4223.0000  4223.0000  4223.0000  4223.0000
mean       2667.3849    3010.0217   259.5315     1.4526    99.3436
std        3446.6457    1560.9407    32.9752     1.8167     3.3630
min         487.3914    1022.5800   217.1990     0.0500    84.5619
25%         750.2271    1830.9900   234.7470     0.1000    98.3491
50%         955.8430    2635.6699   249.5290     0.3400   100.2434
75%        2159.9885    4136.1899   287.6740     2.3900   101.3735
max       14339.2998    6978.6001   326.5880     5.3300   104.1004
